<a href="https://colab.research.google.com/github/wmasfoe/md-editor-models/blob/master/notebooks/train_and_release_t4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="在 Colab 中打开"/></a>

> ⚠️ 本 notebook 使用 **master** 分支的 Qwen3 + Adapter 发布链路。

# 🚀 md-editor 端侧专属小模型: Google Colab L4/T4 矩阵微调与发布

本 Notebook 可在 **Google Colab (推荐 L4 / T4 GPU)** 上一键完成：
1. **环境自检**：检测 NVIDIA L4 / T4 GPU 与硬件加速状态
2. **多模型矩阵微调**：支持单独训练 **0.5B Lite** / **1.5B Standard**，或 **一键全矩阵全自动打包**
3. **自动量化与增量聚合**：转换为 `Q4_K_M` GGUF，并自动增量合并到同一个 `manifest.json`
4. **自动发布**：一键推送到 GitHub Releases 同一版本 Tag 下

In [1]:
#@title ⚙️ [1/4] 配置训练参数与发布模式
#@markdown 选择发布模式与版本（Qwen3-0.6B Lite 基座，支持 Base + 任务 LoRA Adapter）：

mode = "Lite 完整矩阵 (base + gec + completion + distill)" #@param ["Lite 完整矩阵 (base + gec + completion + distill)", "base", "gec adapter", "completion adapter", "distill adapter", "legacy 完整模型(旧版兼容)"]
version_tag = "v1.2.0" #@param {type:"string"}
tier = "lite" #@param ["lite", "standard"]
base_model = "Qwen/Qwen3-0.6B" #@param {type:"string"}
branch = "master" #@param {type:"string"}

# 验证 GPU 状态 (需显示 L4 / T4 / A100)
!nvidia-smi

print(f"🎯 模式: {mode}")
print(f"🏷️ 版本: {version_tag} | Tier: {tier} | Base: {base_model}")

🎯 已选择模式: 1.5B (Standard - L4约15分钟)
🏷️ 版本标签:   v1.0.0
Tue Sep  1 14:28:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N

In [ ]:
#@title 📦 [2/4] 克隆/更新仓库并预装全套依赖环境
import os
if not os.path.exists('/content/md-editor-models'):
    !git clone -b "$branch" https://github.com/wmasfoe/md-editor-models.git /content/md-editor-models
else:
    !git -C /content/md-editor-models fetch origin "$branch" && git -C /content/md-editor-models checkout "$branch" && git -C /content/md-editor-models pull origin "$branch"

%cd /content/md-editor-models
!pip install -q trl peft pangu datasets transformers accelerate sentencepiece gguf protobuf huggingface_hub "torchao>=0.16.0"
print("✅ 仓库与依赖就绪")

Cloning into '/content/md-editor-models'...
remote: Enumerating objects: 119, done.
remote: Counting objects: 100% (119/119), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 119 (delta 50), reused 95 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (119/119), 799.91 KiB | 22.85 MiB/s, done.
Resolving deltas: 100% (50/50), done.
/content/md-editor-models
From https://github.com/wmasfoe/md-editor-models
 * branch            master     -> FETCH_HEAD
Already up to date.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 118.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 47.1 MB/s eta 0:00:00


In [ ]:
#@title 🔑 [3/4] 配置 GitHub Token (用于自动发布 Release)
import os
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
except Exception:
    token = None

if not token and not os.environ.get('GH_TOKEN') and not os.environ.get('GITHUB_TOKEN'):
    token = input("请输入你的 GitHub Token (按回车直接上传): ").strip()

if token:
    os.environ['GH_TOKEN'] = token
    os.environ['GITHUB_TOKEN'] = token
    print("✅ GitHub Token 配置成功！")
else:
    print("ℹ️ 未提供 Token，训练完成后模型将保存在 output 目录。")

In [ ]:
#@title 🚀 [4/4] 启动 Qwen3 Adapter 训练、量化与发布！
!chmod +x scripts/release_model.sh
import subprocess

def run(cmd):
    print(f"\n$ {cmd}")
    result = subprocess.run(cmd, shell=True, executable="/bin/bash")
    if result.returncode != 0:
        raise SystemExit(f"命令失败 (exit {result.returncode}): {cmd}")

task_list = ["gec", "completion", "distill"]

if mode == "Lite 完整矩阵 (base + gec + completion + distill)":
    run(f"./scripts/release_model.sh {version_tag} {base_model} --tier {tier} --asset base")
    for task in task_list:
        run(f"./scripts/release_model.sh {version_tag} {base_model} --tier {tier} --asset adapter --task {task}")
elif mode == "base":
    run(f"./scripts/release_model.sh {version_tag} {base_model} --tier {tier} --asset base")
elif mode == "legacy 完整模型(旧版兼容)":
    run(f"./scripts/release_model.sh {version_tag} {base_model}")
else:
    task = mode.split()[0]  # gec/completion/distill adapter
    run(f"./scripts/release_model.sh {version_tag} {base_model} --tier {tier} --asset adapter --task {task}")

print("\n🎉 全流程执行完毕！模型矩阵已上线 GitHub Releases！")